# TẢI ZIP TN4 — đầy đủ, kèm checkpoint

Notebook **chỉ đọc**. Không train, không xoá, không clone mã.

Lấy **mọi tệp nén của TN4** trên Drive — cả `final.pth` (trọng số model) —
gộp thành một tệp rồi bấm tải về máy.

TN4 gồm bốn cấu hình, mỗi cấu hình ba seed:

| tệp nén Drive | cấu hình |
|---|---|
| `tn4_ds_tcn_c64_k3_..._a0.6_..._seed{0,1,2}` | Ours-64/61 alpha 0,6 |
| `tn4_ds_tcn_c64_k5_..._a0_..._seed{0,1,2}` | Ours-64/121 Pearson thuần ★ |
| `tn4_ds_tcn_c64_k5_..._mse_..._seed{0,1,2}` | Ours-64/121 MSE thuần (nền) |
| `tn4_ds_tcn_c192_k5_..._a0.2_..._seed{0,1,2}` | Ours-192/121 alpha 0,2 |

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Liệt kê mọi tệp nén TN4

In [ ]:
import glob, os, subprocess

SRC = "/content/drive/MyDrive/mobivital"
zips = sorted(p for p in glob.glob(SRC + "/tn4_*.zip"))

tong = 0
print("  %-72s %8s   %s" % ("tệp", "MB", "sửa lúc"))
print("  " + "-" * 100)
for p in zips:
    mb = os.path.getsize(p) / 1048576
    tong += mb
    gio = subprocess.run(["date", "-r", p, "+%m-%d %H:%M"],
                         capture_output=True, text=True).stdout.strip()
    print("  %-72s %8.1f   %s" % (os.path.basename(p)[:72], mb, gio))
print("  " + "-" * 100)
print("  %d tệp  ·  tổng %.0f MB" % (len(zips), tong))

## 3. Gộp NGUYÊN VẸN và SẮP THEO CẤU HÌNH

Giải mọi zip, gộp các bản trùng (kiểm khớp byte), rồi xếp lại đúng layout
`runs/tn4/` trong repo:

```
TN4_day_du/
   64-121__pearson/   64-61__hybrid-a0.6/   64-121__mse/   192-121__hybrid-a0.2/
      seed0/  seed1/  seed2/
         final.pth   curve.csv   scores.csv   selection.txt
   summary.csv
```

In [ ]:
import shutil, tarfile, tempfile, hashlib, csv, glob

def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()

# config_id đầy đủ  ->  tên thư mục ngắn
MAP = {
 "ds_tcn_c64_k3_n4_none_do0.2_dpel_mse_pearson_a0.6_corr0.9":  "64-61__hybrid-a0.6",
 "ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_pearson_a0_corr0.9":    "64-121__pearson",
 "ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9":               "64-121__mse",
 "ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_pearson_a0.2_corr0.9": "192-121__hybrid-a0.2",
}
def tach(run_id):
    for cfg in MAP:
        if run_id.startswith(cfg + "_seed"):
            return MAP[cfg], run_id[len(cfg) + 1:]      # -> (tên ngắn, "seedN")
    raise ValueError("không map được " + run_id)

raw = tempfile.mkdtemp()
for p in zips:
    subprocess.run(["unzip", "-oq", p, "-d",
                    os.path.join(raw, os.path.splitext(os.path.basename(p))[0])],
                   check=True)

work = tempfile.mkdtemp()
out_root = os.path.join(work, "TN4_day_du")
os.makedirs(out_root)

# gom mọi run_id (thư mục *_seedN nằm trong .../tn4/)
run_ids = sorted({os.path.basename(d) for d in glob.glob(raw + "/*/tn4/*")
                  if os.path.isdir(d) and "_seed" in os.path.basename(d)})

xung_dot = []
def chep(src, dst):
    if os.path.exists(dst):
        if sha(src) != sha(dst):
            xung_dot.append(dst)
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

n_pth = 0
for rid in run_ids:
    short, seed = tach(rid)
    dst_dir = os.path.join(out_root, short, seed)
    for src in glob.glob(raw + "/*/tn4/" + rid + "/final.pth"):
        chep(src, os.path.join(dst_dir, "final.pth")); n_pth += 1 if not os.path.exists(os.path.join(dst_dir,"final.pth")) else 0
    for src in glob.glob(raw + "/*/tn4/" + rid + "/curve.csv"):
        chep(src, os.path.join(dst_dir, "curve.csv"))
    for src in glob.glob(raw + "/*/tn4/scores_" + rid + ".csv"):
        chep(src, os.path.join(dst_dir, "scores.csv"))
    for src in glob.glob(raw + "/*/tn4/" + rid + ".txt"):
        chep(src, os.path.join(dst_dir, "selection.txt"))

# summary.csv: gộp mọi dòng, khử trùng theo run_id
rows, hdr = {}, None
for f in glob.glob(raw + "/*/tn4/summary.csv"):
    for r in csv.DictReader(open(f)):
        if hdr is None: hdr = list(r.keys())
        rows[r["run_id"]] = r
with open(os.path.join(out_root, "summary.csv"), "w", newline="") as fo:
    w = csv.DictWriter(fo, fieldnames=hdr); w.writeheader()
    for k in sorted(rows): w.writerow(rows[k])

n_pth = len(glob.glob(out_root + "/*/*/final.pth"))
n_file = sum(len(fs) for _, _, fs in os.walk(out_root))

archive = "/content/TN4_day_du.tar.gz"
with tarfile.open(archive, "w:gz") as tf:
    tf.add(out_root, arcname="TN4_day_du")

mb = os.path.getsize(archive) / 1048576
print("  %d cấu hình · %d file (%d final.pth)" % (len(MAP), n_file, n_pth))
if xung_dot:
    print("  !! %d file trùng nhưng KHÁC byte:" % len(xung_dot))
    for x in xung_dot[:5]: print("     ", x)
else:
    print("  mọi bản trùng khớp byte")
print("  -> %s   %.1f MB" % (archive, mb))

## 4. Xem cây thư mục

In [ ]:
print(subprocess.run(["bash", "-lc",
      "cd %s && find TN4_day_du | sort" % work],
      capture_output=True, text=True).stdout)

## 5. Tải về máy

In [ ]:
from google.colab import files
files.download("/content/TN4_day_du.tar.gz")

## 6. Ở máy

```bash
tar -xzf TN4_day_du.tar.gz
```

Mỗi tệp nén một thư mục, mỗi thư mục có `tn4/<run_id>/final.pth`.